In [1]:
import sys
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
PyTorch: 2.11.0+cu130
CUDA disponível: True
GPU: Tesla T4


In [4]:
!pip install -q -U transformers datasets accelerate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 40.6 MB/s eta 0:00:00


In [3]:
!git clone https://github.com/juliaduboisas/plnEp1.git

Cloning into 'plnEp1'...
remote: Enumerating objects: 88, done.
remote: Counting objects: 100% (88/88), done.
remote: Compressing objects: 100% (71/71), done.
remote: Total 88 (delta 30), reused 63 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (88/88), 14.54 MiB | 6.23 MiB/s, done.
Resolving deltas: 100% (30/30), done.


In [5]:
%cd plnEp1
!ls

/content/plnEp1
baselineModel.ipynb	     tf_idf_reglog.ipynb
bertimbauModel.ipynb	     tfidfregord.ipynb
dataExploration.ipynb	     tfidf_sublinear.ipynb
ep1-enunciado.pdf	     TF_IDF_Word_+_Linear_SVM.ipynb
exponentialregression.ipynb  TF_IDF_+_XGBoost.ipynb
laya_custom		     ../data/train.csv
layaFineTune.ipynb	     ../data/train_noDuplicates.csv
pipelines		     ../data/train_noSameClassDuplicates.csv
README.md		     useModel.ipynb
roBERTaModel.ipynb	     word2vecmlp.ipynb
sbertModel.ipynb	     word2vecreglog.ipynb


In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("../data/train.csv")

print(df.shape)
print(df.head())
print(df["clarity"].value_counts())

(20092, 2)
                                           resp_text clarity
0   Prezado(a) Senhor(a),   Esclarecemos que o Se...      c5
1   Prezada cidadã,  As informações sobre óbitos ...      c1
2   Prezado Senhor Julio,  A Ouvidoria-Geral da P...      c1
3   Prezado(a) Senhor(a),   Esclarecemos que o Se...    c234
4   Senhor, O Serviço de Informações ao Cidadão d...    c234
clarity
c5      6892
c234    6853
c1      6347
Name: count, dtype: int64


In [7]:
X = df["resp_text"]
y = df["clarity"]

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=123,
    stratify=y
)

print("Treino:", len(X_train))
print("Validação:", len(X_val))

Treino: 16073
Validação: 4019


In [8]:
from transformers import AutoTokenizer

MODEL_NAME = "neuralmind/bert-base-portuguese-cased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/210k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [9]:
texto_exemplo = X_train.iloc[0]

tokens = tokenizer.tokenize(texto_exemplo)

print("Texto:")
print(texto_exemplo)

print("\nTokens:")
print(tokens[:50])

Texto:
 Informo que a PROGEP respondeu ao seu pedido de informação, como segue:  "Prezada, não possui código livres. Aproveitamos para informar que há concurso vigente para este cargo. " 

Tokens:
['Inf', '##orm', '##o', 'que', 'a', 'PR', '##O', '##GE', '##P', 'respondeu', 'ao', 'seu', 'pedido', 'de', 'informação', ',', 'como', 'segue', ':', '"', 'Pre', '##za', '##da', ',', 'não', 'possui', 'código', 'livres', '.', 'Apro', '##ve', '##itam', '##os', 'para', 'informar', 'que', 'há', 'concurso', 'vigente', 'para', 'este', 'cargo', '.', '"']


In [10]:
encoded = tokenizer(
    texto_exemplo,
    truncation=True,
    max_length=256,
    padding="max_length"
)

print(encoded.keys())

KeysView({'input_ids': [101, 5027, 19560, 22280, 179, 123, 11635, 22317, 6993, 22310, 9396, 320, 347, 4794, 125, 4205, 117, 271, 5229, 131, 107, 4407, 852, 285, 117, 346, 1776, 5552, 8216, 119, 14173, 301, 7327, 128, 221, 21424, 179, 1307, 5826, 18886, 221, 860, 2466, 119, 107, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [11]:
print("input_ids:")
print(encoded["input_ids"][:30])

print("\nattention_mask:")
print(encoded["attention_mask"][:30])

input_ids:
[101, 5027, 19560, 22280, 179, 123, 11635, 22317, 6993, 22310, 9396, 320, 347, 4794, 125, 4205, 117, 271, 5229, 131, 107, 4407, 852, 285, 117, 346, 1776, 5552, 8216, 119]

attention_mask:
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [12]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3
)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  438MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from th

In [13]:
label2id = {
    "c1": 0,
    "c234": 1,
    "c5": 2
}

id2label = {
    0: "c1",
    1: "c234",
    2: "c5"
}

In [17]:
y_train_encoded = y_train.map(label2id)
y_val_encoded = y_val.map(label2id)

print(y_train_encoded.value_counts())
print(y_val_encoded.value_counts())

clarity
2    5513
1    5482
0    5078
Name: count, dtype: int64
clarity
2    1379
1    1371
0    1269
Name: count, dtype: int64


In [18]:
truncation=True
max_length=256

def tokenize_function(texts):
    return tokenizer(
        texts.tolist(),
        truncation=True,
        max_length=256
    )

train_encodings = tokenize_function(X_train)
val_encodings = tokenize_function(X_val)

In [19]:
from datasets import Dataset

train_dataset = Dataset.from_dict({
    "input_ids": train_encodings["input_ids"],
    "attention_mask": train_encodings["attention_mask"],
    "labels": y_train_encoded.tolist()
})

val_dataset = Dataset.from_dict({
    "input_ids": val_encodings["input_ids"],
    "attention_mask": val_encodings["attention_mask"],
    "labels": y_val_encoded.tolist()
})

print(train_dataset)
print(val_dataset)

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 16073
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 4019
})


In [20]:
model.config.label2id = label2id
model.config.id2label = id2label

print(model.config.label2id)
print(model.config.id2label)

{'c1': 0, 'c234': 1, 'c5': 2}
{0: 'c1', 1: 'c234', 2: 'c5'}


In [21]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

# ==================================================
# 1. Padding dinâmico
# ==================================================

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)


# ==================================================
# 2. Métricas
# ==================================================

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)

    f1_macro = f1_score(
        labels,
        predictions,
        average="macro"
    )

    return {
        "accuracy": accuracy,
        "f1_macro": f1_macro
    }


# ==================================================
# 3. Configuração do treinamento
# ==================================================

training_args = TrainingArguments(
    output_dir="./bertimbau_results",

    num_train_epochs=2,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    learning_rate=2e-5,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,

    logging_steps=100,

    save_total_limit=1,

    report_to="none"
)


# ==================================================
# 4. Trainer
# ==================================================

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)

print("Trainer configurado com sucesso!")

Trainer configurado com sucesso!


In [ ]:
# ==================================================
# MINI-TREINO DE TESTE
# ==================================================

# Pequenas amostras apenas para testar o pipeline
train_small = train_dataset.select(range(500))
val_small = val_dataset.select(range(100))

print("Treino do teste:", len(train_small))
print("Validação do teste:", len(val_small))


# --------------------------------------------------
# Criamos um Trainer separado para o teste
# --------------------------------------------------

mini_training_args = TrainingArguments(
    output_dir="./bertimbau_mini_test",

    # Apenas 1 época para o teste
    num_train_epochs=1,

    # Batch pequeno porque estamos em CPU
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    learning_rate=2e-5,

    # Avaliar ao final
    eval_strategy="epoch",

    # Não precisamos salvar checkpoints desse teste
    save_strategy="no",

    logging_steps=10,

    report_to="none"
)


mini_trainer = Trainer(
    model=model,
    args=mini_training_args,

    train_dataset=train_small,
    eval_dataset=val_small,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)


# --------------------------------------------------
# Treinamento
# --------------------------------------------------

print("Iniciando mini-treino...")

mini_trainer.train()

print("Mini-treino terminado!")

Treino do teste: 500
Validação do teste: 100
Iniciando mini-treino...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.103177,1.079663,0.430000,0.335592


Mini-treino terminado!


In [22]:
# ==================================================
# BERTIMBAU — CONFIGURAÇÃO DO TREINAMENTO
# ==================================================

from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

# --------------------------------------------------
# Recarrega o BERTimbau original
# --------------------------------------------------
# Importante: o modelo usado no mini-treino foi alterado.
# Aqui começamos novamente do modelo pré-treinado original.

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3
)

model.config.label2id = label2id
model.config.id2label = id2label


# --------------------------------------------------
# Configuração do treinamento
# --------------------------------------------------

training_args = TrainingArguments(
    output_dir="./bertimbau_results",

    num_train_epochs=2,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    learning_rate=2e-5,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,

    logging_steps=100,
    save_total_limit=1,

    # Aproveita a GPU Tesla T4
    fp16=True,

    report_to="none"
)


# --------------------------------------------------
# Trainer
# --------------------------------------------------

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)

print("BERTimbau e Trainer configurados.")
print("Treino:", len(train_dataset))
print("Validação:", len(val_dataset))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from th

BERTimbau e Trainer configurados.
Treino: 16073
Validação: 4019


In [ ]:
# ==================================================
# TREINAMENTO
# ==================================================

print("Iniciando treinamento do BERTimbau...")

trainer.train()

print("Treinamento concluído!")

Iniciando treinamento do BERTimbau...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.031283,1.028269,0.460562,0.457560
2,0.972206,1.028803,0.465539,0.458712


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Treinamento concluído!


In [ ]:
results = trainer.evaluate()
print(results)

Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro
0.972206,1.028803,2,0.465539,0.458712


{'eval_loss': 1.0288033485412598, 'eval_accuracy': 0.4655386912167206, 'eval_f1_macro': 0.45871237952547866}


In [ ]:
# ==================================================
# EXPERIMENTO 2 — LEARNING RATE = 3e-5
# ==================================================

from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

# Recomeça do BERTimbau pré-treinado original
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3
)

model.config.label2id = label2id
model.config.id2label = id2label


training_args_exp2 = TrainingArguments(
    output_dir="./bertimbau_exp2",

    num_train_epochs=2,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    learning_rate=3e-5,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,

    logging_steps=100,
    save_total_limit=1,

    fp16=True,
    report_to="none"
)


trainer_exp2 = Trainer(
    model=model,
    args=training_args_exp2,

    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    data_collator=data_collator,
    compute_metrics=compute_metrics
)


print("Iniciando Experimento 2...")

trainer_exp2.train()

results_exp2 = trainer_exp2.evaluate()

print("\nResultado do Experimento 2:")
print(results_exp2)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from th

Iniciando Experimento 2...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.031303,1.026201,0.458323,0.445611
2,0.955180,1.027391,0.468525,0.464856


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro
0.955180,1.027391,2,0.468525,0.464856



Resultado do Experimento 2:
{'eval_loss': 1.027390718460083, 'eval_accuracy': 0.46852450858422495, 'eval_f1_macro': 0.46485643502857626}


In [ ]:
# ==================================================
# EXPERIMENTO 3 — LEARNING RATE = 5e-5
# ==================================================

from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

# Recomeça do BERTimbau pré-treinado original
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3
)

model.config.label2id = label2id
model.config.id2label = id2label


training_args_exp3 = TrainingArguments(
    output_dir="./bertimbau_exp3",

    num_train_epochs=2,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    learning_rate=5e-5,  # única mudança em relação ao Exp. 2

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,

    logging_steps=100,
    save_total_limit=1,

    fp16=True,
    report_to="none"
)


trainer_exp3 = Trainer(
    model=model,
    args=training_args_exp3,

    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    data_collator=data_collator,
    compute_metrics=compute_metrics
)


print("Iniciando Experimento 3...")

trainer_exp3.train()

results_exp3 = trainer_exp3.evaluate()

print("\nResultado do Experimento 3:")
print(results_exp3)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from th

Iniciando Experimento 3...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.046778,1.041067,0.446877,0.431076
2,0.984234,1.030839,0.456332,0.454008


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro
0.984234,1.030839,2,0.456332,0.454008



Resultado do Experimento 3:
{'eval_loss': 1.030839204788208, 'eval_accuracy': 0.45633242100024884, 'eval_f1_macro': 0.4540083446182623}


In [ ]:
# ==================================================
# EXPERIMENTO 4 — 3 ÉPOCAS + LEARNING RATE = 3e-5
# ==================================================

from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3
)

model.config.label2id = label2id
model.config.id2label = id2label

training_args_exp4 = TrainingArguments(
    output_dir="./bertimbau_exp4",

    num_train_epochs=3,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    learning_rate=3e-5,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,

    logging_steps=100,
    save_total_limit=1,

    fp16=True,
    report_to="none"
)

trainer_exp4 = Trainer(
    model=model,
    args=training_args_exp4,

    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Iniciando Experimento 4...")

trainer_exp4.train()

results_exp4 = trainer_exp4.evaluate()

print("\nResultado do Experimento 4:")
print(results_exp4)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from th

Iniciando Experimento 4...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.041576,1.033675,0.460562,0.443898
2,0.979122,1.044088,0.468276,0.445824
3,0.844270,1.115784,0.464792,0.465652


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro
0.844270,1.115784,3,0.464792,0.465652



Resultado do Experimento 4:
{'eval_loss': 1.115783929824829, 'eval_accuracy': 0.4647922368748445, 'eval_f1_macro': 0.46565157930128587}


In [ ]:
# Verifica quantos textos ultrapassam 256 tokens

token_lengths = [
    len(tokenizer.encode(text, add_special_tokens=True))
    for text in df["resp_text"]
]

print("Total de textos:", len(token_lengths))
print("Textos com mais de 256 tokens:", sum(x > 256 for x in token_lengths))
print(
    "Percentual truncado em 256:",
    f"{100 * sum(x > 256 for x in token_lengths) / len(token_lengths):.2f}%"
)

print("Maior quantidade de tokens:", max(token_lengths))
print("Mediana:", np.median(token_lengths))
print("Percentil 90:", np.percentile(token_lengths, 90))
print("Percentil 95:", np.percentile(token_lengths, 95))

Total de textos: 20092
Textos com mais de 256 tokens: 7204
Percentual truncado em 256: 35.86%
Maior quantidade de tokens: 4429
Mediana: 182.0
Percentil 90: 534.9000000000015
Percentil 95: 719.0


In [ ]:
# ==================================================
# EXPERIMENTO 5 — MAX_LENGTH = 512
# ==================================================

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from datasets import Dataset

# --------------------------------------------------
# Tokenização com até 512 tokens
# --------------------------------------------------

tokenizer_512 = AutoTokenizer.from_pretrained(MODEL_NAME)

train_encodings_512 = tokenizer_512(
    X_train.tolist(),
    truncation=True,
    max_length=512
)

val_encodings_512 = tokenizer_512(
    X_val.tolist(),
    truncation=True,
    max_length=512
)


# --------------------------------------------------
# Datasets
# --------------------------------------------------

train_dataset_512 = Dataset.from_dict({
    "input_ids": train_encodings_512["input_ids"],
    "attention_mask": train_encodings_512["attention_mask"],
    "labels": y_train_encoded.tolist()
})

val_dataset_512 = Dataset.from_dict({
    "input_ids": val_encodings_512["input_ids"],
    "attention_mask": val_encodings_512["attention_mask"],
    "labels": y_val_encoded.tolist()
})


# --------------------------------------------------
# Modelo novo
# --------------------------------------------------

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3
)

model.config.label2id = label2id
model.config.id2label = id2label


# --------------------------------------------------
# Padding dinâmico
# --------------------------------------------------

data_collator_512 = DataCollatorWithPadding(
    tokenizer=tokenizer_512
)


# --------------------------------------------------
# Treinamento
# --------------------------------------------------

training_args_exp5 = TrainingArguments(
    output_dir="./bertimbau_exp5",

    num_train_epochs=2,

    # 512 tokens consomem mais memória
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    # Batch efetivo = 4 × 2 = 8
    gradient_accumulation_steps=2,

    learning_rate=3e-5,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,

    logging_steps=100,
    save_total_limit=1,

    fp16=True,
    report_to="none"
)


trainer_exp5 = Trainer(
    model=model,
    args=training_args_exp5,

    train_dataset=train_dataset_512,
    eval_dataset=val_dataset_512,

    data_collator=data_collator_512,
    compute_metrics=compute_metrics
)


print("Iniciando Experimento 5...")
trainer_exp5.train()

results_exp5 = trainer_exp5.evaluate()

print("\nResultado do Experimento 5:")
print(results_exp5)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from th

Iniciando Experimento 5...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,2.069346,1.036088,0.461309,0.457063
2,1.944728,1.030148,0.464792,0.460694


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro
1.944728,1.030148,2,0.464792,0.460694



Resultado do Experimento 5:
{'eval_loss': 1.0301483869552612, 'eval_accuracy': 0.4647922368748445, 'eval_f1_macro': 0.4606941994130922}


In [23]:
# ==================================================
# EXPERIMENTO 6 — CONFIGURAÇÃO DE REFERÊNCIA
# BERTimbau | max_length=256 | lr=3e-5 | 2 épocas
# ==================================================

import numpy as np
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    set_seed
)

# --------------------------------------------------
# Reprodutibilidade
# --------------------------------------------------

SEED = 123
set_seed(SEED)


# --------------------------------------------------
# Modelo novo, inicializado após definir a seed
# --------------------------------------------------

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3
)

model.config.label2id = label2id
model.config.id2label = id2label


# --------------------------------------------------
# Configuração do treinamento
# --------------------------------------------------

training_args_exp6 = TrainingArguments(
    output_dir="./bertimbau_exp6",

    num_train_epochs=2,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    learning_rate=3e-5,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,

    logging_steps=100,
    save_total_limit=1,

    fp16=True,

    seed=SEED,
    data_seed=SEED,

    report_to="none"
)


# --------------------------------------------------
# Trainer
# --------------------------------------------------

trainer_exp6 = Trainer(
    model=model,
    args=training_args_exp6,

    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    data_collator=data_collator,
    compute_metrics=compute_metrics
)


# --------------------------------------------------
# Treinamento
# --------------------------------------------------

print("Iniciando Experimento 6...")

trainer_exp6.train()


# --------------------------------------------------
# Avaliação
# --------------------------------------------------

results_exp6 = trainer_exp6.evaluate()

print("\nResultado do Experimento 6:")
print(results_exp6)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from th

Iniciando Experimento 6...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.048970,1.049494,0.449614,0.449779
2,0.982515,1.037106,0.467280,0.466031


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro
0.982515,1.037106,2,0.467280,0.466031



Resultado do Experimento 6:
{'eval_loss': 1.0371062755584717, 'eval_accuracy': 0.46728041801443143, 'eval_f1_macro': 0.4660308326931803}


In [24]:
# ==================================================
# ANÁLISE DOS ERROS
# ==================================================

predictions = trainer_exp6.predict(val_dataset)

y_true = predictions.label_ids
y_pred = np.argmax(predictions.predictions, axis=-1)

print("Classification Report:\n")

print(
    classification_report(
        y_true,
        y_pred,
        target_names=["c1", "c234", "c5"],
        digits=4
    )
)

print("\nMatriz de confusão:\n")

cm = confusion_matrix(y_true, y_pred)

print(cm)

Classification Report:

              precision    recall  f1-score   support

          c1     0.4782    0.5177    0.4972      1269
        c234     0.3953    0.3596    0.3766      1371
          c5     0.5207    0.5279    0.5243      1379

    accuracy                         0.4673      4019
   macro avg     0.4648    0.4684    0.4660      4019
weighted avg     0.4645    0.4673    0.4654      4019


Matriz de confusão:

[[657 358 254]
 [462 493 416]
 [255 396 728]]


In [25]:
# ============================================================
# VALIDAÇÃO CRUZADA — BERTimbau
# Configuração de referência (Exp. 6)
# ============================================================

import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed
)


# ------------------------------------------------------------
# 1. Configuração
# ------------------------------------------------------------

MODEL_NAME = "neuralmind/bert-base-portuguese-cased"

SEED = 123
N_SPLITS = 3

MAX_LENGTH = 256
LEARNING_RATE = 3e-5
EPOCHS = 2
BATCH_SIZE = 8


# ------------------------------------------------------------
# 2. Dados
# ------------------------------------------------------------

df = pd.read_csv("../data/train.csv")

X = df["resp_text"].reset_index(drop=True)
y = df["clarity"].reset_index(drop=True)

label2id = {
    "c1": 0,
    "c234": 1,
    "c5": 2
}


# ------------------------------------------------------------
# 3. Tokenizer
# ------------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)


# ------------------------------------------------------------
# 4. Métricas
# ------------------------------------------------------------

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    return {
        "accuracy": accuracy_score(
            labels,
            predictions
        ),
        "f1_macro": f1_score(
            labels,
            predictions,
            average="macro"
        )
    }


# ------------------------------------------------------------
# 5. Cross-validation
# ------------------------------------------------------------

cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)

fold_results = []


for fold, (train_idx, val_idx) in enumerate(
    cv.split(X, y),
    start=1
):

    print("\n" + "=" * 60)
    print(f"FOLD {fold}/{N_SPLITS}")
    print("=" * 60)

    set_seed(SEED)

    # --------------------------------------------------------
    # Dados do fold
    # --------------------------------------------------------

    X_train = X.iloc[train_idx]
    X_val = X.iloc[val_idx]

    y_train = y.iloc[train_idx].map(label2id)
    y_val = y.iloc[val_idx].map(label2id)


    # --------------------------------------------------------
    # Tokenização
    # --------------------------------------------------------

    train_encodings = tokenizer(
        X_train.tolist(),
        truncation=True,
        max_length=MAX_LENGTH
    )

    val_encodings = tokenizer(
        X_val.tolist(),
        truncation=True,
        max_length=MAX_LENGTH
    )


    # --------------------------------------------------------
    # Datasets
    # --------------------------------------------------------

    train_dataset = Dataset.from_dict({
        "input_ids": train_encodings["input_ids"],
        "attention_mask": train_encodings["attention_mask"],
        "labels": y_train.tolist()
    })

    val_dataset = Dataset.from_dict({
        "input_ids": val_encodings["input_ids"],
        "attention_mask": val_encodings["attention_mask"],
        "labels": y_val.tolist()
    })


    # --------------------------------------------------------
    # Modelo novo para cada fold
    # --------------------------------------------------------

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=3
    )

    model.config.label2id = label2id
    model.config.id2label = {
        0: "c1",
        1: "c234",
        2: "c5"
    }


    # --------------------------------------------------------
    # Treinamento
    # --------------------------------------------------------

    training_args = TrainingArguments(
        output_dir=f"./bertimbau_cv_fold_{fold}",

        num_train_epochs=EPOCHS,

        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,

        learning_rate=LEARNING_RATE,

        eval_strategy="epoch",
        save_strategy="epoch",

        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,

        save_total_limit=1,

        fp16=torch.cuda.is_available(),

        seed=SEED,
        data_seed=SEED,

        report_to="none"
    )


    trainer = Trainer(
        model=model,
        args=training_args,

        train_dataset=train_dataset,
        eval_dataset=val_dataset,

        data_collator=data_collator,
        compute_metrics=compute_metrics
    )


    # --------------------------------------------------------
    # Treinamento + avaliação
    # --------------------------------------------------------

    trainer.train()

    result = trainer.evaluate()

    fold_results.append({
        "fold": fold,
        "f1_macro": result["eval_f1_macro"],
        "accuracy": result["eval_accuracy"]
    })

    print(
        f"Fold {fold}: "
        f"F1 Macro = {result['eval_f1_macro']:.4f} | "
        f"Accuracy = {result['eval_accuracy']:.4f}"
    )


# ------------------------------------------------------------
# 6. Resultado final
# ------------------------------------------------------------

results_df = pd.DataFrame(fold_results)

print("\n" + "=" * 60)
print("RESULTADO DA VALIDAÇÃO CRUZADA")
print("=" * 60)

print(results_df)

print(
    f"\nMacro F1: "
    f"{results_df['f1_macro'].mean():.4f} "
    f"± {results_df['f1_macro'].std():.4f}"
)

print(
    f"Accuracy: "
    f"{results_df['accuracy'].mean():.4f} "
    f"± {results_df['accuracy'].std():.4f}"
)


FOLD 1/3


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from th

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.053497,1.062613,0.439385,0.420729
2,0.958458,1.053349,0.467602,0.467515


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.061693,1.042172,0.445423,0.442624
2,0.986019,1.053855,0.454532,0.450613


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro
0.986019,1.053855,2,0.454532,0.450613


Fold 3: F1 Macro = 0.4506 | Accuracy = 0.4545

RESULTADO DA VALIDAÇÃO CRUZADA
   fold  f1_macro  accuracy
0     1  0.467515  0.467602
1     2  0.466139  0.466179
2     3  0.450613  0.454532

Macro F1: 0.4614 ± 0.0094
Accuracy: 0.4628 ± 0.0072


In [ ]:
# ==================================================
# ANÁLISE DOS ERROS — EXPERIMENTO 6
# ==================================================

import pandas as pd
import numpy as np

# Predições do modelo de referência
predictions = trainer_exp6.predict(val_dataset)

y_true = predictions.label_ids
y_pred = np.argmax(predictions.predictions, axis=-1)

# Recupera os textos correspondentes à validação
error_df = pd.DataFrame({
    "text": X_val.reset_index(drop=True),
    "true_label": y_true,
    "predicted_label": y_pred
})

# Converte os IDs de volta para os nomes das classes
error_df["true_label"] = error_df["true_label"].map(id2label)
error_df["predicted_label"] = error_df["predicted_label"].map(id2label)

# Mantém apenas os exemplos classificados incorretamente
errors = error_df[
    error_df["true_label"] != error_df["predicted_label"]
].copy()

print("Total de exemplos de validação:", len(error_df))
print("Total de erros:", len(errors))
print(
    "Taxa de erro:",
    f"{100 * len(errors) / len(error_df):.2f}%"
)

print("\nDistribuição dos tipos de erro:")
print(
    errors
    .groupby(["true_label", "predicted_label"])
    .size()
    .sort_values(ascending=False)
)


Total de exemplos de validação: 4019
Total de erros: 2128
Taxa de erro: 52.95%

Distribuição dos tipos de erro:
true_label  predicted_label
c234        c1                 458
c5          c234               416
c234        c5                 413
c1          c234               359
            c5                 246
c5          c1                 236
dtype: int64


In [ ]:
# Mostra alguns exemplos de cada tipo de erro
for true_label, predicted_label in [
    ("c1", "c234"),
    ("c1", "c5"),
    ("c234", "c1"),
    ("c234", "c5"),
    ("c5", "c1"),
    ("c5", "c234")
]:
    subset = errors[
        (errors["true_label"] == true_label) &
        (errors["predicted_label"] == predicted_label)
    ]

    print("\n" + "=" * 80)
    print(f"REAL: {true_label}  →  PREVISTO: {predicted_label}")
    print("=" * 80)

    for _, row in subset.head(5).iterrows():
        print("\n", row["text"])


REAL: c1  →  PREVISTO: c234

  Senhor Carlos,  O Serviço de Informações ao Cidadão do Ministério do Planejamento, Desenvolvimento e Gestão agradece o seu contato.  Em atenção à sua solicitação apresentamos em anexo as planilhas com a relação nominal dos servidores públicos federais inativos e pensionistas do  Estado do Rio do Janeiro, do poder executivo federal.  Esclarecemos que não faz parte das competências deste Ministério a Gestão de Pessoas e Administração dos Dados relativos a outros Poderes, e aos Estados e Municípios, razão pela qual não é possível a apresentação de tais dados.  Atenciosamente  Responsável pela resposta: Coordenação-Geral de Atendimento e Suporte Destinatário do recurso de primeira instância: Diretor do Departamento de Gestão de Sistemas de Pessoal - DGSIS Ministério do Planejamento, Desenvolvimento e Gestão (MP) 

  Prezado Paulo Cícero Sousa, Informamos que para dúvidas relacionadas a disponibilidade de vaga para docentes a fim de aproveitamento de concurso

In [ ]:
# ==================================================
# EXPERIMENTO 7 — WEIGHT DECAY = 0.01
# Configuração do Exp. 6 + regularização
# ==================================================

from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    set_seed
)

SEED = 123
set_seed(SEED)

# --------------------------------------------------
# Modelo novo
# --------------------------------------------------

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3
)

model.config.label2id = label2id
model.config.id2label = id2label


# --------------------------------------------------
# Configuração
# --------------------------------------------------

training_args_exp7 = TrainingArguments(
    output_dir="./bertimbau_exp7",

    num_train_epochs=2,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    learning_rate=3e-5,

    # Nova variável sendo testada
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,

    logging_steps=100,
    save_total_limit=1,

    fp16=True,

    seed=SEED,
    data_seed=SEED,

    report_to="none"
)


# --------------------------------------------------
# Trainer
# --------------------------------------------------

trainer_exp7 = Trainer(
    model=model,
    args=training_args_exp7,

    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    data_collator=data_collator,
    compute_metrics=compute_metrics
)


# --------------------------------------------------
# Treinamento + avaliação
# --------------------------------------------------

print("Iniciando Experimento 7...")

trainer_exp7.train()

results_exp7 = trainer_exp7.evaluate()

print("\nResultado do Experimento 7:")
print(results_exp7)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from th

Iniciando Experimento 7...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.050532,1.056874,0.434685,0.433282
2,0.978191,1.034899,0.467280,0.465337


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro
0.978191,1.034899,2,0.467280,0.465337



Resultado do Experimento 7:
{'eval_loss': 1.034899115562439, 'eval_accuracy': 0.46728041801443143, 'eval_f1_macro': 0.46533665488082293}


In [ ]:
# ==================================================
# EXPERIMENTO 8 — PESO MAIOR PARA c234
# ==================================================

import torch
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    set_seed
)

SEED = 123
set_seed(SEED)


# --------------------------------------------------
# Modelo novo
# --------------------------------------------------

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3
)

model.config.label2id = label2id
model.config.id2label = id2label


# --------------------------------------------------
# Trainer com loss ponderada
# --------------------------------------------------

class WeightedLossTrainer(Trainer):

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None
    ):
        labels = inputs.pop("labels")

        outputs = model(**inputs)
        logits = outputs.logits

        # Pesos:
        # c1 = 1.0
        # c234 = 1.25
        # c5 = 1.0
        weights = torch.tensor(
            [1.0, 1.25, 1.0],
            device=logits.device
        )

        loss_function = torch.nn.CrossEntropyLoss(
            weight=weights
        )

        loss = loss_function(logits, labels)

        if return_outputs:
            return loss, outputs

        return loss


# --------------------------------------------------
# Configuração do treinamento
# --------------------------------------------------

training_args_exp8 = TrainingArguments(
    output_dir="./bertimbau_exp8",

    num_train_epochs=2,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    learning_rate=3e-5,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,

    logging_steps=100,
    save_total_limit=1,

    fp16=True,

    seed=SEED,
    data_seed=SEED,

    report_to="none"
)


# --------------------------------------------------
# Treinamento
# --------------------------------------------------

trainer_exp8 = WeightedLossTrainer(
    model=model,
    args=training_args_exp8,

    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    data_collator=data_collator,
    compute_metrics=compute_metrics
)


print("Iniciando Experimento 8...")

trainer_exp8.train()

results_exp8 = trainer_exp8.evaluate()

print("\nResultado do Experimento 8:")
print(results_exp8)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from th

Iniciando Experimento 8...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.048475,1.050237,0.409803,0.355564
2,0.984717,1.036502,0.460562,0.462775


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro
0.984717,1.036502,2,0.460562,0.462775



Resultado do Experimento 8:
{'eval_loss': 1.0365021228790283, 'eval_accuracy': 0.46056232893754667, 'eval_f1_macro': 0.4627745090617657}
